<a href="https://colab.research.google.com/github/tendolkarurja/IPD_Finance/blob/main/IPD_Fin_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df_2022 = pd.read_csv('/content/drive/MyDrive/IPD_sentiments/economic_times_headlines_2022.csv')
df_2023 = pd.read_csv('/content/drive/MyDrive/IPD_sentiments/economic_times_headlines_2023.csv')
df_2024 = pd.read_csv('/content/drive/MyDrive/IPD_sentiments/economic_times_headlines_2024.csv')
df_2025 = pd.read_csv('/content/drive/MyDrive/IPD_sentiments/economic_times_headlines_2025.csv')


In [ ]:
print(df_2022.info())
print(df_2023.info())
print(df_2024.info())
print(df_2025.info())

In [ ]:
df_news = pd.concat([df_2022, df_2023, df_2024, df_2025], axis = 0)
df_news.reset_index(drop=True)

In [ ]:
noise_keywords = ['/bollywood', '/lifestyle', '/panache', '/entertainment', '/magazines', '/astrology', '/horoscope', '/sports']

In [ ]:
occ_noise = {i:0 for i in noise_keywords}

for keyword in noise_keywords:
    occ_noise[keyword] += df_news['Headline link'].str.contains(keyword).sum()

In [ ]:
occ_noise

In [ ]:
df_news.info()

In [ ]:
news_volume_per_day = df_news.groupby('Date').size()
print(news_volume_per_day)

min_news = news_volume_per_day.min()
print(f"The lowest number of headlines on any day was: {min_news}")

In [ ]:
(df_news['Date'] == '29-12-2024').unique()

In [ ]:
df_news['Date'] = pd.to_datetime(df_news['Date'], dayfirst=True)
full_range = pd.date_range(start='2022-01-01', end='2025-06-11')

present_dates = pd.to_datetime(df_news['Date'].unique())
missing_dates = full_range.difference(present_dates)

print(f"Total missing days: {len(missing_dates)}")
print(missing_dates)

In [ ]:
for_2025 = pd.date_range(start='2025-01-01', end='2025-06-11')

missing_dates = for_2025.difference(present_dates)

print(f"Total missing days in 2025: {len(missing_dates)}")
print(missing_dates)

In [ ]:
for keyword in noise_keywords:
    df_noise = df_news[df_news['Headline link'].str.contains(keyword)]
    print(df_noise.groupby('Date').size())

In [ ]:
df_news.drop('Archive', axis =1, inplace = True)

In [ ]:
df_news

In [ ]:
pattern = '|'.join(i for i in noise_keywords)
df_news_clean = df_news[~df_news['Headline link'].str.contains(pattern, case=False, na=False)].copy()

# 3. Reset index to keep it clean (0 to N)
df_news_clean = df_news_clean.reset_index(drop=True)

print(f"Original Records: {len(df_news)}")
print(f"Records after Noise Removal: {len(df_news_clean)}")
print(f"Removed: {len(df_news) - len(df_news_clean)} rows.")

In [ ]:
df_news_clean['Headline link'].unique()

In [ ]:
df_news_clean['Headline'][1]

In [ ]:
df_news_clean.shape

In [ ]:
nifty_companies = [
    'RELIANCE', 'HDFC', 'ICICI', 'INFOSYS', 'TCS', 'WIPRO',
    'HINDUNILVR', 'ITC', 'SBIN', 'BHARTI', 'LT', 'AXIS',
    'KOTAK', 'MARUTI', 'ASIAN PAINTS', 'SUN PHARMA', 'RELIANCE', 'ADANI', 'TATA POWER', 'NTPC', 'ONGC', 'JSW'
]

In [ ]:
ticker_patterns = r'\b(' + '|'.join(nifty_companies) + r')\b'

In [ ]:
df_news_new=df_news_clean

In [ ]:
df_news_new['ticker_match'] = df_news_new['Headline'].str.contains(ticker_patterns, case=False, na=False)

print(df_news_new['ticker_match'].sum())

In [ ]:
import re

# Find all matches in each headline
matches = df_news_new['Headline'].str.findall(ticker_patterns, flags=re.IGNORECASE)

# Convert list of matches into rows
ticker_counts = matches.explode().str.upper().value_counts()

print(ticker_counts)

In [ ]:
df_news_new.info()

In [ ]:
df_ticker = df_news_new[df_news_new['ticker_match']].copy()

In [ ]:
df_ticker.shape

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [ ]:
model_name = "yiyanghkust/finbert-esg"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
labels = ["Environmental", "Social", "Governance", "None"]

def classify_esg(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs).item()

    return labels[pred]

In [ ]:
df_ticker['esg_label'] = df_ticker['Headline'].apply(classify_esg)

In [ ]:
df_esg = df_ticker[df_ticker['esg_label'] != "None"].copy()

In [ ]:
print(df_esg['esg_label'].value_counts())

In [ ]:
import re

matches = df_esg['Headline'].str.findall(ticker_patterns, flags=re.IGNORECASE)

df_esg['ticker'] = matches

ticker_esg_counts = (
    df_esg.explode('ticker')
    .groupby(['ticker', 'esg_label'])
    .size()
    .unstack(fill_value=0)
)

print(ticker_esg_counts)

In [ ]:
print(df_esg.shape,
df_ticker.shape)

In [ ]:
df_esg.info()

In [ ]:
model_name = "yiyanghkust/finbert-tone"

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer_sent = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model_sent = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")

sent_labels = ["Negative", "Neutral", "Positive"]

In [ ]:
def classify_sentiment_batch(texts):
    inputs = tokenizer_sent(texts, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model_sent(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    preds = torch.argmax(probs, dim=1).tolist()
    confidences = torch.max(probs, dim=1).values.tolist()

    labels = [sent_labels[p] for p in preds]

    return labels, confidences

In [ ]:
batch_size = 32

sentiments = []
conf_scores = []

texts = df_esg['Headline'].tolist()

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    labels, confs = classify_sentiment_batch(batch)

    sentiments.extend(labels)
    conf_scores.extend(confs)

df_esg['sentiment'] = sentiments
df_esg['confidence'] = conf_scores

In [ ]:
score_map = {
    "Positive": 1,
    "Neutral": 0,
    "Negative": -1
}

df_esg['sentiment_score'] = df_esg['sentiment'].map(score_map)

In [ ]:
df_esg['weighted_score'] = df_esg['sentiment_score'] * df_esg['confidence']

In [ ]:
df_esg

In [ ]:
df_breakdown = (
    df_esg.explode('ticker')
    .groupby(['ticker', 'esg_label', 'sentiment'])
    .size()
    .unstack(fill_value=0)
)

print(df_breakdown)

In [ ]:
base_path = "/content/drive/MyDrive/IPD_sentiments"

import os
os.makedirs(base_path, exist_ok=True)

In [ ]:
model_sent.save_pretrained(f"{base_path}/finbert_sentiment_model")
tokenizer_sent.save_pretrained(f"{base_path}/finbert_sentiment_model")

In [ ]:
model.save_pretrained(f"{base_path}/finbert_esg_model")
tokenizer.save_pretrained(f"{base_path}/finbert_esg_model")

In [ ]:
df_esg.to_csv(f"{base_path}/df_esg.csv", index=False)

In [ ]:
## LOADING NEXT TIME DIRECTLY PRETRAINED MODELS

# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# model_sent = AutoModelForSequenceClassification.from_pretrained(f"{base_path}/finbert_sentiment_model")
# tokenizer_sent = AutoTokenizer.from_pretrained(f"{base_path}/finbert_sentiment_model")

# model_esg = AutoModelForSequenceClassification.from_pretrained(f"{base_path}/finbert_esg_model")
# tokenizer_esg = AutoTokenizer.from_pretrained(f"{base_path}/finbert_esg_model")

In [ ]:
# LOADING ESG DATASET

df_esg = pd.read_csv(f"{base_path}/df_esg.csv")

In [ ]:
# APPLYING XAI

In [ ]:
!pip install shap

In [ ]:
def predict_proba(texts):
    # ✅ ensure proper format
    texts = [str(t) for t in texts]

    inputs = tokenizer_sent(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model_sent(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1).numpy()
    return probs

In [ ]:
import shap

explainer = shap.Explainer(predict_proba, tokenizer_sent)

In [ ]:
text = df_esg['Headline'].iloc[0]

shap_values = explainer([text])

In [ ]:
shap_values = explainer([text])

shap.plots.text(shap_values[0])